# GameForge3D — Phase 2: Asset Category Router (DistilBERT)

**FYP 2026-2027 | NUML Dept. of Computer Science**  
**Supervisor:** Ms. Tooba Sagheer

This notebook fine-tunes a **DistilBERT** model to classify text prompts into 4 categories:  
`Weapon` | `Vehicle` | `Prop` | `Creature`

**Steps:**
1. Mount Google Drive
2. Install Libraries
3. Load train / val splits
4. Tokenize dataset
5. Define DistilBERT model
6. Train with HuggingFace Trainer
7. Evaluate on val set
8. Save checkpoint to Drive
9. Quick inference test

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_DIR       = '/content/drive/MyDrive/GameForge3D/data'
CHECKPOINT_DIR = '/content/drive/MyDrive/GameForge3D/checkpoints/router'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print('Drive mounted.')
print(f'Data dir       : {DATA_DIR}')
print(f'Checkpoint dir : {CHECKPOINT_DIR}')

## Step 2 — Install Libraries

In [ ]:
!pip install -q transformers datasets scikit-learn pandas torch
print('Libraries ready.')

## Step 3 — Load Train / Val Splits

These CSVs were saved to Drive in Phase 1.

In [ ]:
import pandas as pd

df_train = pd.read_csv(f'{DATA_DIR}/router_train.csv')
df_val   = pd.read_csv(f'{DATA_DIR}/router_val.csv')

print(f'Train: {len(df_train)} samples')
print(f'Val  : {len(df_val)} samples')
print()
print('Train label distribution:')
print(df_train['label'].value_counts())

## Step 4 — Label Encoding & Tokenization

In [ ]:
import torch
from torch.utils.data import Dataset
from transformers import DistilBertTokenizerFast

# --- Label map ---
LABELS    = ['Creature', 'Prop', 'Vehicle', 'Weapon']
LABEL2ID  = {l: i for i, l in enumerate(LABELS)}
ID2LABEL  = {i: l for l, i in LABEL2ID.items()}
NUM_LABELS = len(LABELS)

print('Label map:', LABEL2ID)

# --- Tokenizer ---
MODEL_NAME = 'distilbert-base-uncased'
tokenizer  = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)
print(f'Tokenizer loaded: {MODEL_NAME}')

# --- Custom Dataset ---
class RouterDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=64):
        self.encodings = tokenizer(
            df['prompt'].tolist(),
            truncation=True,
            padding='max_length',
            max_length=max_len,
            return_tensors='pt'
        )
        self.labels = torch.tensor(
            [LABEL2ID[l] for l in df['label']], dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels':         self.labels[idx]
        }

train_dataset = RouterDataset(df_train, tokenizer)
val_dataset   = RouterDataset(df_val,   tokenizer)

print(f'Train dataset size : {len(train_dataset)}')
print(f'Val   dataset size : {len(val_dataset)}')

## Step 5 — Load DistilBERT Classification Model

In [ ]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = model.to(device)
print(f'Model loaded on: {device}')
print(f'Parameters     : {sum(p.numel() for p in model.parameters()):,}')

## Step 6 — Train with HuggingFace Trainer

> **Expected time: ~3-5 minutes on T4 GPU**

In [ ]:
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_macro': f1_score(labels, preds, average='macro')
    }

training_args = TrainingArguments(
    output_dir                  = '/content/router_training',
    num_train_epochs            = 8,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 16,
    learning_rate               = 2e-5,
    weight_decay                = 0.01,
    warmup_ratio                = 0.1,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1_macro',
    logging_steps               = 10,
    report_to                   = 'none',
    fp16                        = torch.cuda.is_available()
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    compute_metrics = compute_metrics
)

print('Starting training...')
trainer.train()
print('Training complete!')

## Step 7 — Evaluate on Validation Set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

results = trainer.evaluate()
print('\n=== Validation Results ===')
for k, v in results.items():
    print(f'  {k}: {v:.4f}')

# --- Full classification report ---
preds_out = trainer.predict(val_dataset)
preds     = np.argmax(preds_out.predictions, axis=1)
true      = preds_out.label_ids

print('\n=== Classification Report ===')
print(classification_report(true, preds, target_names=LABELS))

# --- Confusion matrix ---
cm = confusion_matrix(true, preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=LABELS,
            yticklabels=LABELS, cmap='Blues')
plt.title('Confusion Matrix — Asset Router (Val Set)')
plt.ylabel('True')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig(f'{DATA_DIR}/router_confusion_matrix.png', dpi=150)
plt.show()
print('Confusion matrix saved.')

## Step 8 — Save Checkpoint to Google Drive

In [ ]:
import shutil

# Save model + tokenizer
SAVE_PATH = f'{CHECKPOINT_DIR}/distilbert_router_v1'
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

# Save label map
import json
with open(f'{SAVE_PATH}/label_map.json', 'w') as f:
    json.dump({'label2id': LABEL2ID, 'id2label': ID2LABEL}, f, indent=2)

print(f'Model saved to: {SAVE_PATH}')
print(f'Files:')
for fn in os.listdir(SAVE_PATH):
    size = os.path.getsize(f'{SAVE_PATH}/{fn}') // 1024
    print(f'  {fn:40s} {size:>6} KB')

## Step 9 — Quick Inference Test

Test the trained router with sample prompts.

In [ ]:
from transformers import pipeline

classifier = pipeline(
    'text-classification',
    model=SAVE_PATH,
    tokenizer=SAVE_PATH,
    device=0 if torch.cuda.is_available() else -1
)

TEST_PROMPTS = [
    'a flaming battle axe with runes',
    'a futuristic hover car',
    'a dragon breathing fire',
    'a wooden barrel with iron bands',
    'a magic wand with glowing tip',
    'a pirate ship with black sails',
    'a zombie rising from the grave',
    'an ancient stone altar'
]

print('=== Inference Test ===')
print(f'{"Prompt":<45} {"Predicted":<12} {"Score"}')
print('-' * 70)
for prompt in TEST_PROMPTS:
    result = classifier(prompt)[0]
    label  = result['label']
    score  = result['score']
    print(f'{prompt:<45} {label:<12} {score:.3f}')

## Phase 2 Complete! ✅

| Output | Location |
|--------|----------|
| Trained Router Model | `checkpoints/router/distilbert_router_v1/` |
| Tokenizer | `checkpoints/router/distilbert_router_v1/` |
| Label Map | `checkpoints/router/distilbert_router_v1/label_map.json` |
| Confusion Matrix | `data/router_confusion_matrix.png` |

**Next → Phase 3: 3D Shape Generator (Shap-E style)**